In [ ]:
import igraph
import squidpy as sq
import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
samples = ['XM-6','T640T1','T1088','T1982','KHE007','T1489','T943','T554','T1821','T2008','KHE008','T1739','T487','T1706','T1768','T1879','T924','T1440','T143','T701','T516','KHE004','T876','T1668','T640T2','T877','T937','T1057','T976']

In [ ]:
sample = "T1668"

In [ ]:
adata = sc.read_h5ad(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")

In [ ]:
adata_sub = sc.read_h5ad(f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad")

In [ ]:
adata_sub.obs.index = adata.obs.index

In [ ]:
adata.obs["louvain_labels"] = adata_sub.obs["louvain_labels"]

In [ ]:
adata.obs

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="louvain_labels", method="wilcoxon")

In [ ]:
rank_results = adata.uns['rank_genes_groups']
 
all_groups_results = pd.DataFrame()
 
group_labels = adata.obs[rank_results['params']['groupby']].unique()
for group_label in group_labels:
    group_df = sc.get.rank_genes_groups_df(adata, group=group_label)
    group_df = group_df.sort_values(by="scores", ascending=False)
    group_df = group_df[abs(group_df['logfoldchanges']) >= 0.25]
    group_df['group'] = group_label
    all_groups_results = pd.concat([all_groups_results, group_df], ignore_index=True)
 
all_groups_results.to_csv(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_de.csv", index=False)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata,color = "louvain_labels", legend_loc='on data',show =False)
plt.savefig(f"/cluster3/yflu/STS/TMA/separated/{sample}/umap.pdf", format="pdf", bbox_inches="tight")

In [ ]:
sc.pl.umap(adata,color = ["NOTCH3"])
sc.pl.violin(adata, ["NOTCH3"], groupby='louvain_labels')

In [ ]:
sc.pl.spatial(adata,color = ["louvain_labels"],spot_size=10)

In [ ]:
adata.obs['celltype']=adata.obs.louvain_labels.copy()
adata.obs.celltype.replace(['1'],'Tumor cells',inplace=True)
adata.obs.celltype.replace(['2'],'Adipocytes',inplace=True)
adata.obs.celltype.replace(['3'],'Endothelial',inplace=True)
adata.obs.celltype.replace(['4'],'Pericytes',inplace=True)
adata.obs.celltype.replace(['5'],'Mono/macro',inplace=True)
adata.obs['celltype_1']=adata.obs.louvain_labels.copy()
adata.obs.celltype_1.replace(['1'],'Tumor cells',inplace=True)
adata.obs.celltype_1.replace(['2'],'Adipocytes',inplace=True)
adata.obs.celltype_1.replace(['3'],'Endothelial',inplace=True)
adata.obs.celltype_1.replace(['4'],'Pericytes',inplace=True)
adata.obs.celltype_1.replace(['5'],'Mono/macro',inplace=True)

In [ ]:
sc.pl.umap(adata, color=["celltype", "celltype_1"], show=False)
adata.obs["celltype"].to_csv(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_celltype.csv")
adata.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_annotated.h5ad")

In [ ]:
sc.pl.umap(adata, color=["celltype", "celltype_1"], show=False)
plt.savefig(f"/cluster3/yflu/STS/TMA/separated/{sample}/umap_celltype.pdf", format="pdf", bbox_inches="tight")
sc.pl.spatial(adata,color = ["celltype","celltype_1"],spot_size=10, show=False)
plt.savefig(f"/cluster3/yflu/STS/TMA/separated/{sample}/spatial_celltype.pdf", format="pdf", bbox_inches="tight")
sc.pl.spatial(adata,color = ["banksy_domain","IFIT1"],spot_size=10, show=False)
plt.savefig(f"/cluster3/yflu/STS/TMA/separated/{sample}/spatial_domain.pdf", format="pdf", bbox_inches="tight")

In [ ]:
samples = ['XM-6','T640T1','T1088','T1982','KHE007','T1489','T943','T554','T1821','T2008','KHE008','T1739','T487','T1706','T1768','T1879','T924','T1440','T143','T701','T516','KHE004','T876','T640T2','T877','T937','T1057','T976']
for sample in samples:
    adata=sc.read_h5ad(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_annotated.h5ad")
    adata.obs["barcode"] = adata.obs["cell_id"]
    adata.obs["barcodekey"] = adata.obs.index
    adata.obs[["celltype","barcode","barcodekey"]].to_csv(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_celltype.csv")

In [ ]:
adata.obs.index